# Azure Key Vault Secrets Sync with Terraform and WIF

Terraform configures Vault as the OIDC issuer and creates a Microsoft Entra federated identity credential. No client secret is created or stored.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=".env")

subscription_id = os.environ["AZURE_SUBSCRIPTION_ID"]
tenant_id = os.environ["AZURE_TENANT_ID"]
os.environ["ARM_SUBSCRIPTION_ID"] = subscription_id
os.environ["ARM_TENANT_ID"] = tenant_id
os.environ["TF_VAR_azure_subscription_id"] = subscription_id
os.environ["TF_VAR_public_oidc_issuer_url"] = "https://vault.jose-merchan.sbx.hashidemos.io"


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((p / ".env" for p in (Path.cwd(), *Path.cwd().parents) if (p / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find .env")
load_dotenv(ENV_FILE)

## Authenticate and confirm the Azure subscription

In [ ]:
! az login --tenant $ARM_TENANT_ID --subscription $ARM_SUBSCRIPTION_ID
! az account show --query '{subscription:name,subscriptionId:id,tenantId:tenantId}' --output table
! az provider register --namespace Microsoft.KeyVault --subscription $ARM_SUBSCRIPTION_ID --wait
! az provider show --namespace Microsoft.KeyVault --subscription $ARM_SUBSCRIPTION_ID --query '{namespace:namespace,state:registrationState}' --output table

## Verify the public Vault OIDC endpoints

In [ ]:
! curl -fsS $TF_VAR_public_oidc_issuer_url/v1/identity/oidc/secrets-sync/.well-known/openid-configuration | jq
! curl -fsS $TF_VAR_public_oidc_issuer_url/v1/identity/oidc/secrets-sync/.well-known/keys | jq

## Initialize and validate

In [ ]:
! terraform -chdir=terraform-azure-wif init
! terraform -chdir=terraform-azure-wif validate

## Review the issuer, audience and subject

In [ ]:
! terraform -chdir=terraform-azure-wif plan

## Apply

The configuration waits 60 seconds for the federated credential and RBAC assignment to propagate before creating the Vault destination.

In [ ]:
! terraform -chdir=terraform-azure-wif apply -auto-approve

## Verify the published issuer and JWKS after apply

In [ ]:
! curl -fsS $TF_VAR_public_oidc_issuer_url/v1/identity/oidc/secrets-sync/.well-known/openid-configuration | jq
! curl -fsS $TF_VAR_public_oidc_issuer_url/v1/identity/oidc/secrets-sync/.well-known/keys | jq

## Verify Vault and Azure Key Vault

In [ ]:
! terraform -chdir=terraform-azure-wif output
! vault read sys/sync/destinations/azure-kv/mapfre-wif-azure-kv
! vault read -format=json sys/sync/destinations/azure-kv/mapfre-wif-azure-kv/associations | jq

In [ ]:
%%bash
KEY_VAULT_NAME=$(terraform -chdir=terraform-azure-wif output -raw key_vault_name)
SECRET_NAME=$(terraform -chdir=terraform-azure-wif output -raw external_secret_name)
az keyvault secret show \
  --vault-name "$KEY_VAULT_NAME" \
  --name "$SECRET_NAME" \
  --query '{name:name,enabled:attributes.enabled,updated:attributes.updated}' \
  --output json

# CLEAN UP

In [ ]:
! terraform -chdir=terraform-azure-wif destroy -auto-approve